## import libraries

In [106]:
import ee
from dotenv import load_dotenv
import os
import ee
import pandas as pd
from pathlib import Path
import pickle

## initialize google earth engine

In [107]:
ee.Authenticate()

True

In [108]:
load_dotenv()  
project = os.getenv("PROJECT")
ee.Initialize(project=project)

## read datasets

In [109]:
root_embeddings = Path("_data/exports/embeddings_exports")
root_dataset = Path("_data/exports/crop_country_exports")

In [110]:
def build_files_dataframe(embeddings, dataset):
    rows = []

    folders = [dataset, embeddings]

    for folder in folders:
        for file in folder.glob("*"):
    
            if file.suffix not in [".csv", ".parquet"]:
                continue
            name = file.stem  
            parts = name.split("_")

            if "embedding" in parts:
                year = parts[-2]
                country = parts[-3]
                crop = "_".join(parts[:-3])
                file_type = "embedding"
            else:
                year = parts[-1]
                country = parts[-2]
                crop = "_".join(parts[:-2])
                file_type = "dataset"
            
            rows.append({
                "path": str(file),
                "crop": crop,
                "country": country,
                "year": int(year),
                "type": file_type
            })

    return pd.DataFrame(rows)

In [111]:
files = build_files_dataframe(root_embeddings, root_dataset)
files

,path,crop,country,year,type
0,_data/exports/crop_country_exports/maize_corn_...,maize_corn_popcorn,pt,2018,dataset
1,_data/exports/crop_country_exports/maize_corn_...,maize_corn_popcorn,pt,2019,dataset
2,_data/exports/crop_country_exports/potatoes_bg...,potatoes,bg,2018,dataset
3,_data/exports/crop_country_exports/winter_barl...,winter_barley,dk,2018,dataset
4,_data/exports/crop_country_exports/winter_barl...,winter_barley,be,2016,dataset
...,...,...,...,...,...
75,_data/exports/embeddings_exports/winter_barley...,winter_barley,bg,2019,embedding
76,_data/exports/embeddings_exports/maize_corn_po...,maize_corn_popcorn,be,2019,embedding
77,_data/exports/embeddings_exports/winter_barley...,winter_barley,ee,2019,embedding
78,_data/exports/embeddings_exports/potatoes_at_2...,potatoes,at,2019,embedding


In [112]:
# pairing embeddings with datasets
df_datasets = files[files['type'] == 'dataset'].drop(columns=['type'])
df_embeddings = files[files['type'] == 'embedding'].drop(columns=['type'])

pairs = pd.merge(df_datasets, 
                 df_embeddings, 
                 on=['crop', 'country', 'year'], 
                 suffixes=('_dataset', '_embedding'))

pairs

,path_dataset,crop,country,year,path_embedding
0,_data/exports/crop_country_exports/maize_corn_...,maize_corn_popcorn,pt,2019,_data/exports/embeddings_exports/maize_corn_po...
1,_data/exports/crop_country_exports/winter_barl...,winter_barley,dk,2019,_data/exports/embeddings_exports/winter_barley...
2,_data/exports/crop_country_exports/potatoes_bg...,potatoes,bg,2019,_data/exports/embeddings_exports/potatoes_bg_2...
3,_data/exports/crop_country_exports/maize_corn_...,maize_corn_popcorn,bg,2019,_data/exports/embeddings_exports/maize_corn_po...
4,_data/exports/crop_country_exports/potatoes_pt...,potatoes,pt,2019,_data/exports/embeddings_exports/potatoes_pt_2...
5,_data/exports/crop_country_exports/potatoes_dk...,potatoes,dk,2019,_data/exports/embeddings_exports/potatoes_dk_2...
6,_data/exports/crop_country_exports/winter_barl...,winter_barley,bg,2019,_data/exports/embeddings_exports/winter_barley...
7,_data/exports/crop_country_exports/maize_corn_...,maize_corn_popcorn,dk,2019,_data/exports/embeddings_exports/maize_corn_po...
8,_data/exports/crop_country_exports/maize_corn_...,maize_corn_popcorn,be,2019,_data/exports/embeddings_exports/maize_corn_po...
9,_data/exports/crop_country_exports/potatoes_ie...,potatoes,ie,2019,_data/exports/embeddings_exports/potatoes_ie_2...


In [113]:
datasets_dict = {}
embeddings_dict = {}

for _, row in pairs.iterrows():
    key = f"{row['crop']}_{row['country']}_{row['year']}"
    
    datasets_dict[key] = pd.read_csv(row['path_dataset'])
    embeddings_dict[key] = pd.read_parquet(row['path_embedding'])

print(f"Successfully loaded {len(datasets_dict)} pairs.")
print(f"Dictionaries keys are  {datasets_dict.keys()} pairs.")

Successfully loaded 20 pairs.
Dictionaries keys are  dict_keys(['maize_corn_popcorn_pt_2019', 'winter_barley_dk_2019', 'potatoes_bg_2019', 'maize_corn_popcorn_bg_2019', 'potatoes_pt_2019', 'potatoes_dk_2019', 'winter_barley_bg_2019', 'maize_corn_popcorn_dk_2019', 'maize_corn_popcorn_be_2019', 'potatoes_ie_2019', 'winter_barley_ee_2019', 'potatoes_at_2019', 'maize_corn_popcorn_at_2019', 'maize_corn_popcorn_ie_2019', 'potatoes_be_2019', 'potatoes_ee_2019', 'winter_barley_at_2019', 'winter_barley_ie_2019', 'winter_barley_be_2019', 'maize_corn_popcorn_ee_2019']) pairs.


In [114]:
datasets_dict['maize_corn_popcorn_pt_2019'].head(2)

,country_id,year,hcat4_code,hcat4_name,long,lat
0,pt,2019,3310106000,maize_corn_popcorn,-8.595169,41.446993
1,pt,2019,3310106000,maize_corn_popcorn,-8.699815,41.345510


In [115]:
embeddings_dict['maize_corn_popcorn_pt_2019'].head(2)

,tile_lon,tile_lat,pixel_row,pixel_col,crs,embedding,long_lat,crop,country_id,year
0,-8.55,41.45,592,43,EPSG:32629,"[6.9150324, -0.11429805, -1.2001295, 4.000432,...","[-8.595169408320043, 41.446993162495055]",maize_corn_popcorn,pt,2019
1,-8.65,41.35,608,3,EPSG:32629,"[5.62293, 0.33403546, 0.2226903, 4.008425, 2.5...","[-8.699815410424158, 41.345509586470214]",maize_corn_popcorn,pt,2019


## for testing reduced dictionaries

In [88]:
reduction_size = len(embeddings_dict['maize_corn_popcorn_pt_2019'])

In [90]:
datasets_dict_reduced = {name: df.iloc[:reduction_size] for name, df in datasets_dict.items()}
embeddings_dict_reduced = {name: df.iloc[:reduction_size] for name, df in embeddings_dict.items()}

## get point images

| Data Type   | Access Method                     | Description                                                |
|--------------|----------------------------------|------------------------------------------------------------|
| Dataset     | `datasets_dict[key].iloc[i]`     | The i-th row of tabular data (labels, etc.).              |
| Embeddings   | `embeddings_dict[key].iloc[i]`   | The i-th row of the embedding vector.                     |
| Satellite    | `gee_dict[key][i]['s2']`         | The full Sentinel-2 time series for that exact point.     |

In [116]:
def get_sits_data(lat, lon, year):
    point = ee.Geometry.Point([lon, lat])
    start = f'{year}-01-01'
    end = f'{year}-12-31'

    s1_bands = ['VV', 'VH']
    s1_col = (ee.ImageCollection("COPERNICUS/S1_GRD")
              .filterBounds(point)
              .filterDate(start, end)
              .filter(ee.Filter.eq('instrumentMode', 'IW'))
              .select(s1_bands))
    
    s2_bands = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
    s2_col = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
              .filterBounds(point)
              .filterDate(start, end)
              #.filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 100))
              .select(s2_bands))
    
    s1_raw = s1_col.getRegion(point, 10).getInfo() # 10 is the resolution
    s2_raw = s2_col.getRegion(point, 10).getInfo() 

    return s1_raw, s2_raw

In [117]:
def to_df(raw_data):
    df = pd.DataFrame(raw_data[1:], columns=raw_data[0])
    df['date'] = pd.to_datetime(df['time'], unit='ms')
    return df.drop(columns=['id', 'time'])

In [128]:
keys = ['maize_corn_popcorn_at_2019', 'maize_corn_popcorn_be_2019', 'maize_corn_popcorn_bg_2019', 'maize_corn_popcorn_dk_2019', 'maize_corn_popcorn_ee_2019', 'maize_corn_popcorn_ie_2019', 'potatoes_at_2019', 'potatoes_be_2019', 'potatoes_bg_2019', 'potatoes_dk_2019', 'potatoes_ee_2019', 'potatoes_ie_2019']
datasets_dict.keys()

dict_keys(['maize_corn_popcorn_pt_2019', 'winter_barley_dk_2019', 'potatoes_bg_2019', 'maize_corn_popcorn_bg_2019', 'potatoes_pt_2019', 'potatoes_dk_2019', 'winter_barley_bg_2019', 'maize_corn_popcorn_dk_2019', 'maize_corn_popcorn_be_2019', 'potatoes_ie_2019', 'winter_barley_ee_2019', 'potatoes_at_2019', 'maize_corn_popcorn_at_2019', 'maize_corn_popcorn_ie_2019', 'potatoes_be_2019', 'potatoes_ee_2019', 'winter_barley_at_2019', 'winter_barley_ie_2019', 'winter_barley_be_2019', 'maize_corn_popcorn_ee_2019'])

In [129]:
keys

['maize_corn_popcorn_at_2019',
 'maize_corn_popcorn_be_2019',
 'maize_corn_popcorn_bg_2019',
 'maize_corn_popcorn_dk_2019',
 'maize_corn_popcorn_ee_2019',
 'maize_corn_popcorn_ie_2019',
 'potatoes_at_2019',
 'potatoes_be_2019',
 'potatoes_bg_2019',
 'potatoes_dk_2019',
 'potatoes_ee_2019',
 'potatoes_ie_2019']

In [130]:
images_dict = {}

for key in keys: # ir fazendo só para alguns
    df_main = datasets_dict[key]
# for key in datasets_dict.keys(): # todos os datasets
#     df_main = datasets_dict[key]
# for key in datasets_dict_reduced.keys(): # para datasets mais pequenos
#     df_main = datasets_dict_reduced[key]
    year = int(key.split('_')[-1])
    
    file_results = []
    
    print(f"Starting extraction for {key} ({len(df_main)} points)...")
    
    for idx, row in df_main.iterrows():
        lat, lon = row['lat'], row['long']
        
        try:
            s1_raw, s2_raw = get_sits_data(lat, lon, year)
            df_s1 = to_df(s1_raw)
            df_s2 = to_df(s2_raw)

            file_results.append({'s1': df_s1, 
                                 's2': df_s2, 
                                 'point_coord': (lon, lat)})
            
        except Exception as e:
            print(f"Error at row {idx} in {key}: {e}")
            file_results.append(None) 
            
    images_dict[key] = file_results

Starting extraction for maize_corn_popcorn_at_2019 (500 points)...
Starting extraction for maize_corn_popcorn_be_2019 (500 points)...
Starting extraction for maize_corn_popcorn_bg_2019 (500 points)...
Starting extraction for maize_corn_popcorn_dk_2019 (500 points)...
Starting extraction for maize_corn_popcorn_ee_2019 (500 points)...
Starting extraction for maize_corn_popcorn_ie_2019 (500 points)...
Starting extraction for potatoes_at_2019 (500 points)...
Starting extraction for potatoes_be_2019 (500 points)...
Starting extraction for potatoes_bg_2019 (500 points)...
Starting extraction for potatoes_dk_2019 (500 points)...
Starting extraction for potatoes_ee_2019 (500 points)...
Starting extraction for potatoes_ie_2019 (500 points)...


In [131]:
images_dict.keys()

dict_keys(['maize_corn_popcorn_at_2019', 'maize_corn_popcorn_be_2019', 'maize_corn_popcorn_bg_2019', 'maize_corn_popcorn_dk_2019', 'maize_corn_popcorn_ee_2019', 'maize_corn_popcorn_ie_2019', 'potatoes_at_2019', 'potatoes_be_2019', 'potatoes_bg_2019', 'potatoes_dk_2019', 'potatoes_ee_2019', 'potatoes_ie_2019'])

In [132]:
# print(f"type(images_dict['winter_barley_dk_2019']) is a {type(images_dict['winter_barley_dk_2019'])}")
# print(f"type(images_dict['winter_barley_dk_2019']) has size {len(images_dict['winter_barley_dk_2019'])}, for each point")
# print(f"type(images_dict['winter_barley_dk_2019']) keys are {images_dict['winter_barley_dk_2019'][0].keys()}")

In [133]:
# images_dict['winter_barley_dk_2019'][0]['point_coord']

## export results

In [134]:
images_output_dir = Path("_data/exports/image_points_exports")
images_output_dir.mkdir(parents=True, exist_ok=True)

def save_gee_data(data_dict, folder):
    for key, content in data_dict.items():
        file_path = folder / f"{key}_image.pkl"
        with open(file_path, 'wb') as f:
            pickle.dump(content, f)
    print(f"Successfully saved {len(data_dict)} files to {folder}")

save_gee_data(images_dict, images_output_dir)

Successfully saved 12 files to _data/exports/image_points_exports


In [69]:
# to load later
with open('_data/exports/image_points_exports/potatoes_dk_2019_image.pkl', 'rb') as f:
    ooo = pickle.load(f)

ooo[0]['point_coord']

(8.969402676915083, 55.97289628204808)